# Birkhoff polytope — full exploration, n = 2, 3

The Birkhoff polytope \(B_n\) is the set of \(n \times n\) doubly stochastic
matrices — nonnegative, every row and column summing to 1 — flattened into
\(\mathbb{R}^{n^2}\). By the Birkhoff–von Neumann theorem, its vertices are
exactly the \(n!\) permutation matrices, and its dimension is \((n-1)^2\)
(matching `birkhoff.sage`). For each \(n\), this notebook:

1. lists all vertices (`birkhoff_vertices` in `common.sage`, reduced to a
   genuinely full-dimensional chart via `reduce_full_dim` — the natural
   \(\mathbb{R}^{n^2}\) embedding has codimension \(2n-1\), too much for
   `reduce_codim1`, which only drops one coordinate),
2. computes the canonical form via the general nbc method (Brown–Dupont
   Prop. 6.7) — \(B_n\) is **not simple** for any \(n \geq 2\) (at a
   permutation-matrix vertex, \(n^2-n\) facets meet, exceeding the
   dimension \((n-1)^2\) by exactly \(n-1\)), so Prop. 6.10 never
   applies here,
3. computes the projective (polar) dual,
4. checks the **volume conjecture** at the centroid,
5. enumerates all triangulations and identifies which are regular,
6. computes the secondary polytope and its vertex embedding.

**This family caps out earlier than almost any other in this catalog** —
\(n!\) vertices means \(B_4\) already has 24 vertices in a 9-dimensional
polytope, and its canonical form alone was measured to take about 80
seconds while building this notebook (vs. well under a second for
\(B_3\)); triangulation enumeration at \(n=4\) didn't finish at all in a
reasonable time. So the full six-step treatment below stops at \(n=3\);
a final section gives \(B_4\)'s f-vector and volume only (both still
cheap — under a second each, since they don't go through the nbc method
or triangulation enumeration).

**A note on the volume numbers**: \(\mathrm{vol}(B_3)=1/8\) and
\(\mathrm{vol}(B_4)=11/11340\) below are Sage's own induced (Euclidean,
relative-to-the-affine-hull) measure — and they match, exactly, the
leading coefficients of Beck & Pixton's Ehrhart polynomials \(H_3(t)\)
and \(H_4(t)\) for these polytopes (*The Ehrhart polynomial of the
Birkhoff polytope*, Discrete Comput. Geom. 30 (2003) 623–637), which is
precisely what Ehrhart theory says that leading coefficient equals: the
volume of the polytope relative to the primitive sublattice of its own
affine hull. **Not** the same normalization as the more commonly quoted
\(\mathrm{vol}(B_3)=9/8\) figure seen elsewhere — that's Beck–Pixton's own
separately-rescaled "\(\mathrm{vol}\, B_n\)" table entry, \(n^{n-1}\)
times the Ehrhart leading coefficient (checked directly against their
paper before writing any of this, not assumed).

**Requires the `sagemath` Jupyter kernel** and must be opened from the
same synced folder as the `.sage` files — see `README.md`.

In [ ]:
load("general_canonical_forms.sage")

That `load` pulls in `common.sage` (`birkhoff_vertices`,
`reduce_full_dim`, `polar_dual`, `secondary_polytope_data`) and
`vertex_sum_canonical_forms.sage` too, and runs
`general_canonical_forms.sage`'s own test suite as a side effect (scroll
up for that PASS/FAIL output). Everything below is fresh, per-\(n\)
exploration specific to this family.

## B_2 — the segment (d = 1)

In [ ]:
n = 2
pts = reduce_full_dim(birkhoff_vertices(n))
d = len(pts[0])
y = [var(f"y{i}") for i in range(1, d + 1)]
P = Polyhedron(vertices=pts)
print(f"{len(pts)} vertices ({factorial(n)} permutation matrices), dimension {P.dimension()} (= (n-1)^2 = {(n-1)**2}):")
pts

### Canonical form (Proposition 6.7, general nbc method)

In [ ]:
phi = general_canonical_form_density(P, y)
verify_pole_structure(f"Birkhoff B_{n}", phi, P, y)
phi

### Canonical form, broken down by vertex

In [ ]:
rows = canonical_form_by_vertex(P, y)
print_canonical_form_by_vertex(rows)

### Projective dual

In [ ]:
Dual = polar_dual(P)
print(f"dual: {Dual.n_vertices()} vertices, {Dual.n_facets()} facets")
Dual.vertices_list()

### Volume conjecture: canonical form vs. the volume of the projective dual (at the centroid)

In [ ]:
centroid = [sum(QQ(v[i]) for v in pts) / len(pts) for i in range(d)]
pts_centered = [tuple(QQ(v[i]) - centroid[i] for i in range(d)) for v in pts]
P_centered = Polyhedron(vertices=pts_centered)

phi_centroid = general_canonical_form_density(P_centered, y)
val_at_centroid = phi_centroid.subs({yi: 0 for yi in y})

vol_dual = Dual.volume()
target = factorial(d) * vol_dual
print("phi at the centroid =", val_at_centroid)
print("d! * Vol(dual) =", target)
match_plus = bool((val_at_centroid - target) == 0)
match_minus = bool((val_at_centroid + target) == 0)
print("matches d! * Vol(dual):", match_plus, " matches -d! * Vol(dual):", match_minus)
assert match_plus or match_minus, "volume-conjecture identity failed -- would be a real bug"

### All triangulations, and which are regular

In [ ]:
sp, sp_reduced, rows = secondary_polytope_data(pts)
for t, gkz, is_reg in rows:
    print(t, "GKZ vector:", gkz, " regular:", is_reg)
print(f"{len(rows)} triangulation(s) total, {sum(1 for _, _, r in rows if r)} regular")

### Secondary polytope: vertex embedding

In [ ]:
print(f"secondary polytope: dimension {sp_reduced.dimension()}, {sp_reduced.n_vertices()} vertex/vertices")
sp_reduced.vertices_list()

## B_3 — the Birkhoff polytope proper (d = 4)

In [ ]:
n = 3
pts = reduce_full_dim(birkhoff_vertices(n))
d = len(pts[0])
y = [var(f"y{i}") for i in range(1, d + 1)]
P = Polyhedron(vertices=pts)
print(f"{len(pts)} vertices ({factorial(n)} permutation matrices), dimension {P.dimension()} (= (n-1)^2 = {(n-1)**2}):")
pts

### Canonical form (Proposition 6.7, general nbc method)

In [ ]:
phi = general_canonical_form_density(P, y)
verify_pole_structure(f"Birkhoff B_{n}", phi, P, y)
phi

### Canonical form, broken down by vertex

In [ ]:
rows = canonical_form_by_vertex(P, y)
print_canonical_form_by_vertex(rows)

### Projective dual

In [ ]:
Dual = polar_dual(P)
print(f"dual: {Dual.n_vertices()} vertices, {Dual.n_facets()} facets")
Dual.vertices_list()

### Volume conjecture: canonical form vs. the volume of the projective dual (at the centroid)

In [ ]:
centroid = [sum(QQ(v[i]) for v in pts) / len(pts) for i in range(d)]
pts_centered = [tuple(QQ(v[i]) - centroid[i] for i in range(d)) for v in pts]
P_centered = Polyhedron(vertices=pts_centered)

phi_centroid = general_canonical_form_density(P_centered, y)
val_at_centroid = phi_centroid.subs({yi: 0 for yi in y})

vol_dual = Dual.volume()
target = factorial(d) * vol_dual
print("phi at the centroid =", val_at_centroid)
print("d! * Vol(dual) =", target)
match_plus = bool((val_at_centroid - target) == 0)
match_minus = bool((val_at_centroid + target) == 0)
print("matches d! * Vol(dual):", match_plus, " matches -d! * Vol(dual):", match_minus)
assert match_plus or match_minus, "volume-conjecture identity failed -- would be a real bug"

### All triangulations, and which are regular

In [ ]:
sp, sp_reduced, rows = secondary_polytope_data(pts)
for t, gkz, is_reg in rows:
    print(t, "GKZ vector:", gkz, " regular:", is_reg)
print(f"{len(rows)} triangulation(s) total, {sum(1 for _, _, r in rows if r)} regular")

### Secondary polytope: vertex embedding

In [ ]:
print(f"secondary polytope: dimension {sp_reduced.dimension()}, {sp_reduced.n_vertices()} vertex/vertices")
sp_reduced.vertices_list()

## B_4 — f-vector and volume only (d = 9)

Canonical form and triangulation enumeration were both measured
infeasible in reasonable time while building this notebook — the
canonical form alone took about 80 seconds (vs. under a second at
\(n=3\)), and triangulation enumeration of 24 points in a 9-dimensional
polytope didn't complete. The f-vector and volume, by contrast, are
cheap: both come directly from Sage's own `Polyhedron` machinery, not
the nbc method or triangulation enumeration, so there's no reason to
skip them.

In [ ]:
n = 4
pts = reduce_full_dim(birkhoff_vertices(n))
P = Polyhedron(vertices=pts)
print(f"{len(pts)} vertices, dimension {P.dimension()}")
print("f-vector:", P.f_vector())
print("volume (induced):", P.volume(measure="induced"))

\(\mathrm{vol}(B_4) = 11/11340\) matches the leading coefficient of
Beck–Pixton's \(H_4(t) = \frac{11}{11340}t^9 + \frac{11}{630}t^8 +
\cdots\) exactly — see the intro cell for why that's the right quantity
to compare against, and not their separately-rescaled table value
\(176/2835\).